In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/all_results.pkl
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/cm_r2_cbfocal.png
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal/fold_2.pt
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal/fold_0.pt
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal/fold_3.pt
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal/fold_4.pt
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal/fold_1.pt
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal/tokenizer/tokenizer.json
/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal/tokenizer/tokenizer_config.json
/kaggle/input/data

In [2]:
# import numpy as np
# import pandas as pd
# import pickle

# # Carregar teste e modelo
# test_df = pd.read_csv('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')

# with open('/kaggle/input/datasets/gabrielsavio/model-final/all_results_r2.pkl', 'rb') as f:
#     obj = pickle.load(f)

# results = obj['bertimbau_large__cb_focal']
# test_probs = results['test_probs']  # (4, 7)

# print(f"CV F1: {results['cv_mean']:.4f} ± {results['cv_std']:.4f}")
# print(f"OOF F1: {results['oof_f1']:.4f}")
# print(f"Test probs:\n{test_probs.round(4)}")

# # Gerar predições
# predictions = test_probs.argmax(axis=1)

# # Submission
# submission = pd.DataFrame({
#     'ID': test_df['ID'],
#     'target': predictions,
# })

# submission.to_csv('/kaggle/working/submission.csv', index=False)
# print(f"\n{submission.to_string(index=False)}")
# print(f"\n✓ submission.csv salvo ({len(submission)} rows)")

In [3]:
# %% [markdown]
# # SPR 2026 – Mammography BI-RADS Classification – Inference
# Notebook de inferência para submissão no Kaggle.
# Carrega pesos treinados (5-fold ensemble) e gera submission.csv.

# %%
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
MAX_LEN     = 192
NUM_CLASSES = 7
N_FOLDS     = 5
BATCH_SIZE  = 16
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Caminho para os pesos (ajustar nome do dataset no Kaggle)
WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs')
if not WEIGHTS_BASE.exists():
    WEIGHTS_BASE = Path('advanced_outputs')

# Config a usar
CONFIG_KEY = 'bertimbau_large__cb_focal'

print(f'Device: {DEVICE}')
print(f'Weights base: {WEIGHTS_BASE}')

# %%
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=192):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item


# %%
# ═══════════════════════════════════════════════════════════════════════════════
# INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        outputs = model(**kwargs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)


# %%
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD TEST DATA
# ═══════════════════════════════════════════════════════════════════════════════
test_path = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
if not test_path.exists():
    test_path = Path('data/test.csv')

test_df = pd.read_csv(test_path)
test_texts = test_df['report'].tolist()

def clean_text(t):
    t = re.sub(r'\s+', ' ', t).strip()
    return t

test_texts = [clean_text(t) for t in test_texts]
print(f'Test samples: {len(test_df)}')

# %%
# ═══════════════════════════════════════════════════════════════════════════════
# RUN ENSEMBLE INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════
weights_dir = WEIGHTS_BASE / 'weights' / CONFIG_KEY
print(f'Loading from: {weights_dir}')

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')

# Dataset & DataLoader
dataset = MammographyDataset(test_texts, tokenizer, MAX_LEN)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=0, pin_memory=True)

test_probs = np.zeros((len(test_df), NUM_CLASSES))
folds_loaded = 0

for fold in range(N_FOLDS):
    weight_path = weights_dir / f'fold_{fold}.pt'
    if not weight_path.exists():
        print(f'Fold {fold} not found, skipping...')
        continue

    print(f'Loading fold {fold}...', end=' ')

    # model = AutoModelForSequenceClassification.from_pretrained(
    #     weights_dir / 'tokenizer',
    #     num_labels=NUM_CLASSES,
    #     ignore_mismatched_sizes=True,
    # ).to(DEVICE)
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(
        '/kaggle/input/datasets/gabrielsavio/model-config/model_config',
        num_labels=NUM_CLASSES,
    )
    model = AutoModelForSequenceClassification.from_config(config).to(DEVICE)

    state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)

    fold_probs = predict(model, loader)
    test_probs += fold_probs
    folds_loaded += 1
    print(f'done (shape: {fold_probs.shape})')

    del model, state_dict
    torch.cuda.empty_cache()

# Média dos folds
test_probs /= folds_loaded
print(f'\n{folds_loaded} folds loaded')

# %%
# ═══════════════════════════════════════════════════════════════════════════════
# SUBMISSION
# ═══════════════════════════════════════════════════════════════════════════════
predictions = test_probs.argmax(axis=1)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'target': predictions,
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'\nsubmission.csv saved ({len(submission)} rows)')

Device: cpu
Weights base: /kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs
Test samples: 4
Loading from: /kaggle/input/datasets/gabrielsavio/model-bi-rads/advanced_outputs/weights/bertimbau_large__cb_focal
Loading fold 0... 

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


done (shape: (4, 7))
Loading fold 1... done (shape: (4, 7))
Loading fold 2... done (shape: (4, 7))
Loading fold 3... done (shape: (4, 7))
Loading fold 4... done (shape: (4, 7))

5 folds loaded

=== SUBMISSION ===
   ID  target
 Acc0       6
 Acc2       2
 Acc4       1
Acc10       2

submission.csv saved (4 rows)
